In [ ]:
import os
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

## Data layout

Assumes data at: `data/chestxray/` with `images/` and `labels.csv` or similar.


In [ ]:
DATA_ROOT = Path('data/chestxray')
IMAGES_DIR = DATA_ROOT / 'images'
LABELS_FILE = DATA_ROOT / 'labels.csv'
print('Images dir:', IMAGES_DIR)

## Transforms

For X-rays, convert to 3-channel by duplicating grayscale channel or use transforms that output 3 channels. Normalization still uses ImageNet stats if using pretrained backbones.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
train_transform = T.Compose([
    T.Resize((256,256)),
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_transform = T.Compose([
    T.Resize((256,256)),
    T.CenterCrop(224),
    T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
print('X-ray transforms ready')

## `ChestXrayDataset` implementation

Minimal dataset parsing a CSV with `image,label` columns.


In [ ]:
class ChestXrayDataset(Dataset):
    def __init__(self, root, images_dir='images', labels_file='labels.csv', transform=None):
        self.root = Path(root)
        self.images_dir = self.root / images_dir
        self.transform = transform
        self.samples = []
        if (self.root / labels_file).exists():
            import csv
            with open(self.root / labels_file, 'r') as fh:
                reader = csv.DictReader(fh)
                for r in reader:
                    img = self.images_dir / r['image']
                    label = int(r.get('label', 0))
                    if img.exists():
                        self.samples.append((str(img), label))
        else:
            for p in sorted(self.images_dir.glob('*')):
                if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.tif']:
                    self.samples.append((str(p), 0))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path)
        # ensure RGB for pretrained backbones
        if img.mode != 'RGB':
            img = img.convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        return img, label

ds = ChestXrayDataset('data/chestxray', transform=val_transform)
print('Samples:', len(ds))
dl = DataLoader(ds, batch_size=8)
batch = next(iter(dl)) if len(ds)>0 else None
if batch is not None:
    imgs, labels = batch
    print('Batch shape:', imgs.shape)